<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [5]</a>'.</span>

In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251110_171343"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "14_preprocess_ptbr"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [3]:
# 2. Carregar dados de entrada
import pandas as pd, os

in_file = os.path.join(PROC_PATH, "news_multisource_clean.parquet")
assert os.path.exists(in_file), f"Arquivo não encontrado: {in_file}. Rode o nb 13 primeiro."

df = pd.read_parquet(in_file)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).copy()

print("Shape:", df.shape)
display(df.head(3))

Shape: (100, 6)


,date,title,text,source,url,origin
0,2025-09-19 07:00:00,Ibovespa bate novo recorde e encosta nos 146 m...,"<a href=""https://news.google.com/rss/articles/...",G1,https://news.google.com/rss/articles/CBMieEFVX...,google_rss
1,2025-09-21 07:00:00,Apenas 4 ações do Ibovespa pagam dividendos ac...,"<a href=""https://news.google.com/rss/articles/...",InfoMoney,https://news.google.com/rss/articles/CBMixAFBV...,google_rss
2,2025-09-30 07:00:00,Veja as 10 maiores altas do Ibovespa em setemb...,"<a href=""https://news.google.com/rss/articles/...",Valor Investe,https://news.google.com/rss/articles/CBMi_AFBV...,google_rss


In [4]:
# Dependência já deve estar instalada via requirements.txt

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [5]:
# 3. Configurações de NLP
import nltk
from nltk.corpus import stopwords
from unidecode import unidecode

nltk.download("stopwords", quiet=True)
STOP_PT = set(stopwords.words("portuguese"))
USE_SPACY = True
try:
    import spacy
    nlp = spacy.load("pt_core_news_lg")
except Exception as exc:
    USE_SPACY = False
    nlp = None
    print(f"spaCy indisponível ({exc}). Usando fallback baseado em regex/nltk.")


ModuleNotFoundError: No module named 'unidecode'

In [ ]:
# 4. Utilitários de limpeza PT-BR
import re

URL_RE   = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w{2,}\b")
NUM_RE   = re.compile(r"\b\d+(?:[.,]\d+)?\b")
WS_RE    = re.compile(r"\s+")

def normalize_whitespace(t: str) -> str:
    return WS_RE.sub(" ", t).strip()

def build_text(row):
    t = (str(row.get("title", "")) + " " + str(row.get("text", ""))).strip()
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    # preserva números? para sentimento, em geral remove:
    t = NUM_RE.sub(" ", t)
    return normalize_whitespace(t)

In [ ]:
# 5. Funções de pré-processamento (spaCy ou fallback)
def preprocess_spacy_pt(text: str):
    doc = nlp(text)
    lemmas = []
    for tok in doc:
        if tok.is_space or tok.is_punct or tok.like_url or tok.like_email:
            continue
        if tok.is_stop:
            continue
        lemma = tok.lemma_.lower().strip()
        if not lemma or len(lemma) < 2:
            continue
        # remove só pontuação residual
        if re.fullmatch(r"[^\wáàâãéêíóôõúç]+", lemma, flags=re.IGNORECASE):
            continue
        lemmas.append(lemma)
    clean = " ".join(lemmas)
    return clean, lemmas

def preprocess_fallback_pt(text: str):
    # regex + NLTK + unidecode (sem lematização)
    tx = text.lower()
    tx = URL_RE.sub(" ", tx)
    tx = EMAIL_RE.sub(" ", tx)
    tx = re.sub(r"[^a-zà-úç\s]", " ", tx, flags=re.IGNORECASE)
    tx = normalize_whitespace(tx)
    tokens = [w for w in tx.split() if w not in STOP_PT and len(w) > 1]
    clean = " ".join(tokens)
    return clean, tokens

In [ ]:
# 6. Aplicar pré-processamento
from tqdm.auto import tqdm
tqdm.pandas()

df = df.assign(text_full=df.apply(build_text, axis=1))

if USE_SPACY:
    out = df["text_full"].progress_apply(preprocess_spacy_pt)
else:
    out = df["text_full"].progress_apply(preprocess_fallback_pt)

df["clean_text"] = out.apply(lambda x: x[0])
df["lemmas"]     = out.apply(lambda x: x[1])
df["n_tokens"]   = df["lemmas"].apply(len)

# Campos auxiliares
df["day"] = df["date"].dt.floor("D")
print("Pré-processamento concluído. Exemplo:")
display(df[["date","source","title","clean_text","n_tokens"]].head(5))

In [ ]:
# 7. Salvar dataset limpo + BOW diário por fonte
import numpy as np

clean_parquet = os.path.join(PROC_PATH, "news_clean.parquet")
clean_csv     = os.path.join(PROC_PATH, "news_clean.csv")

# dataset limpo
cols_out = ["date","day","source","url","origin","title","text","clean_text","lemmas","n_tokens"]
df_out = df[cols_out].copy()
df_out.to_parquet(clean_parquet, index=False)
df_out.to_csv(clean_csv, index=False, encoding="utf-8")
print("✅ Salvo dataset limpo:\n", clean_parquet, "\n", clean_csv)

# BOW diário por fonte (para TF-IDF no nb 15)
bow = (
    df[["day","source","lemmas"]]
    .explode("lemmas")
    .dropna(subset=["lemmas"])
    .groupby(["day","source","lemmas"], as_index=False)
    .size()
    .rename(columns={"size":"tf"})
)

bow_file = os.path.join(PROC_PATH, "bow_daily.parquet")
bow.to_parquet(bow_file, index=False)
print("✅ Salvo BOW diário por fonte:\n", bow_file)
display(bow.head(10))

In [ ]:
# 8. QC rápido (distribuições e amostras)
print("Tamanho do corpus (docs):", df.shape[0])
print("Tokens por doc (p50/p90):",
      int(df["n_tokens"].median()), "/", int(df["n_tokens"].quantile(0.9)))

print("\nTop 15 tokens por origem (amostra):")
top = (
    df[["origin","lemmas"]].explode("lemmas")
      .groupby(["origin","lemmas"]).size()
      .reset_index(name="tf").sort_values(["origin","tf"], ascending=[True, False])
)
for org in top["origin"].unique()[:3]:  # mostra até 3 origens
    subt = top[top["origin"]==org].head(15)
    print(f"\n{org}:")
    print(subt[["lemmas","tf"]].to_string(index=False))
print("\nFeito ✅")